In [124]:
import pandas as pd
import numpy as np

In [125]:
# import the data
path = r'E:\Imarticus_ Learning\Jobs - 2026\Models\data\unzipped'
df = pd.read_csv(path + r'\Life Expectancy Data.csv')
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [126]:
df.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness 5-9 years                 34
 thinness  1-19 years               34
 BMI                                34
Polio                               19
Diphtheria                          19
Life expectancy                     10
Adult Mortality                     10
 HIV/AIDS                            0
Country                              0
Year                                 0
Measles                              0
percentage expenditure               0
infant deaths                        0
Status                               0
under-five deaths                    0
dtype: int64

In [140]:
df.isnull().sum().sort_values(ascending=False)

 thinness 5-9 years       32
 BMI                      32
Polio                     19
Country                    0
Year                       0
Schooling                  0
 HIV/AIDS                  0
Diphtheria                 0
Total expenditure          0
Measles                    0
Hepatitis B                0
percentage expenditure     0
Alcohol                    0
infant deaths              0
Adult Mortality            0
Life expectancy            0
Status                     0
Hepatitis B_missing        0
dtype: int64

In [128]:
# We are droping any null values in Life Expectency column , if treated lead to data leakage
df = df[df['Life expectancy '].notnull()]

In [129]:
df.columns

Index(['Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure',
       'Diphtheria ', ' HIV/AIDS', 'GDP', 'Population',
       ' thinness  1-19 years', ' thinness 5-9 years',
       'Income composition of resources', 'Schooling'],
      dtype='object')

In [130]:
# columns to be dropped
cols = [#'Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       #'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       #'Measles ', ' BMI ', 
       'under-five deaths ', 
       #'Polio', 'Total expenditure',
       #'Diphtheria ', ' HIV/AIDS',
       'GDP','Population', ' thinness  1-19 years',
       #' thinness 5-9 years', 
       'Income composition of resources', 
       #'Schooling'
       ]
df = df.drop(columns=cols)

# these columns wer droped due to multi colinearity problem

In [131]:
# Hepatitis B
# creating a column named HepB_missing which give information to the model 
df['Hepatitis B_missing'] = df['Hepatitis B'].isna().astype(int)
#Fill the missing values with year wise impuation
df['Hepatitis B'] = df.groupby('Year')['Hepatitis B'].transform(lambda x: x.fillna(x.median()))
df['Hepatitis B'] = df['Hepatitis B'].fillna(df['Hepatitis B'].median())


In [132]:
# Total Expenditure

def fill_expenditure(x):
    if x.isna().sum() == 1:
        return x.ffill()

df['Total expenditure'] = df.groupby('Country')['Total expenditure'].transform(fill_expenditure)

# global fallback for any group where even this didn't resolve everything
df['Total expenditure'] = df['Total expenditure'].fillna(df['Total expenditure'].median())

print(df['Total expenditure'].isna().sum())  # confirm 0

0


In [133]:
# ALCOHOL
mask = (df['Alcohol'] == 0.01) & (df['Year'] >= 2012)
df.loc[mask, 'Alcohol'] = np.nan

df['Alcohol'] = df.groupby('Country')['Alcohol'].transform(lambda x: x.ffill().bfill())

df['Alcohol'] = df.groupby(['Status', 'Year'])['Alcohol'].transform(lambda x: x.fillna(x.median())) # for South Sudan


In [134]:
# SCHOOLING

df['Schooling'] = df.groupby(['Status', 'Year'])['Schooling'].transform(
    lambda x: x.fillna(x.median())
)
df['Schooling'] = df.groupby('Status')['Schooling'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
# Diphtheria 
df['Diphtheria '] = df.groupby('Country')['Diphtheria '].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)